# Automatic Language Detection + AI4Bharat IndicConformer
Uploads audio, detects spoken language with SpeechBrain, then transcribes using `ai4bharat/indic-conformer-600m-multilingual`.

> **Note:** IndicConformer itself does not automatically detect language. This notebook performs language identification first.

In [ ]:
!pip -q install -U speechbrain transformers torchaudio librosa soundfile huggingface_hub "onnx==1.20.1" "onnxruntime==1.20.1" "onnxruntime-gpu==1.20.2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.5/291.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 39.1 MB/s eta 0:00:00


In [ ]:
import torch
import torchaudio
import librosa
import soundfile as sf

from google.colab import userdata, files
from huggingface_hub import login
from transformers import AutoModel
from speechbrain.inference.classifiers import EncoderClassifier


In [ ]:
HF_TOKEN = userdata.get("HF_TOKEN")
login(HF_TOKEN)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


Device: cpu


In [ ]:
MODEL_NAME="ai4bharat/indic-conformer-600m-multilingual"

model = AutoModel.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    token=HF_TOKEN
).to(device)

print("IndicConformer loaded.")


config.json:   0%|          | 0.00/241 [00:00<?, ?B/s]

model_onnx.py:   0%|          | 0.00/9.64k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indic-conformer-600m-multilingual:
- model_onnx.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


Please check FRAME_DURATION_MS. The timestamps can be inaccurate
Please check FRAME_DURATION_MS. The timestamps can be inaccurate


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 404 files:   0%|          | 0/404 [00:00<?, ?it/s]

Please check FRAME_DURATION_MS. The timestamps can be inaccurate
IndicConformer loaded.


In [ ]:
lid = EncoderClassifier.from_hparams(
    source="speechbrain/lang-id-voxlingua107-ecapa",
    savedir="pretrained_lid",
    run_opts={"device":device},
)
print("Language detector loaded.")


INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/lang-id-voxlingua107-ecapa' if not cached


hyperparams.yaml:   0%|          | 0.00/1.52k [00:00<?, ?B/s]

INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/lang-id-voxlingua107-ecapa' if not cached


embedding_model.ckpt: reconstructing file:   0%|          |  0.00B / 84.5MB            

embedding_model.ckpt: downloading bytes:           |  0.00B            

...

classifier.ckpt: downloading bytes:           |  0.00B            

INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain/lang-id-voxlingua107-ecapa' if not cached


label_encoder.txt:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, classifier, label_encoder


Language detector loaded.


In [ ]:
LANGUAGE_MAP = {
    "hindi":"hi","english":"en","assamese":"as","bengali":"bn",
    "gujarati":"gu","kannada":"kn","malayalam":"ml","marathi":"mr",
    "nepali":"ne","panjabi":"pa","punjabi":"pa","sanskrit":"sa",
    "sindhi":"sd","tamil":"ta","telugu":"te","urdu":"ur"
}


In [ ]:
uploaded = files.upload()
audio_path = list(uploaded.keys())[0]

audio, sr = librosa.load(audio_path, sr=16000, mono=True)
sf.write("converted.wav", audio, 16000)

out_prob, score, index, label = lid.classify_file("converted.wav")

detected = label[0].split(': ')[1].lower()
language = LANGUAGE_MAP.get(detected)

print("Detected:", detected)
print("Indic language code:", language)

if language is None:
    raise ValueError(
        f"Detected language '{detected}' is not mapped. "
        "Add it to LANGUAGE_MAP or choose a language manually."
    )

wav, sr = torchaudio.load("converted.wav")
wav = wav.to(device)

print("\nRunning transcription...")

# Depending on the repository version, either __call__ or a custom inference
# method may be used. This follows the published API.
result = model(wav, language, "rnnt")

print("\n===== Transcript =====")
print(result)

Saving Assamese - The Two Roads.mp3 to Assamese - The Two Roads.mp3


Detected: assamese
Indic language code: as

Running transcription...

===== Transcript =====
দুটা ৰাস্তা আছে যীচুৱে শিকাইছিল যে দুটা ৰাস্তা আছে প্ৰত্যেকজন মানুহেই পাপৰ বহল ৰাস্তাৰেহে চলিছে যিয়ে বিনাশলৈ পৰিচালনা কৰে আমি সকলোৱে আদমৰ পৰা উত্তৰাধিকাৰী সূত্ৰে পোৱা পাপতে আছো যীচুৱে কৈছিল যে আমি ঠেক বাটেৰে সোমাব লাগে যি বাটেই ঈশ্বৰৰ স্বৰ্গলৈ পৰিচালনা কৰে যদি এজন ব্যক্তিয়ে এই জীৱনত যীচুক বিশ্বাস কৰে আৰু তেওঁৰ অনুগামী হয় তেওঁ অনন্ত জীৱন পায় তেওঁ বহল ৰাস্তা এৰি ঠেক পটেৰে চলে যি পটে স্বৰ্গলৈ পৰিচালনা কৰে ঠেক পটেৰে চলা সকলৰ কাৰণে আৰু পাপৰ দণ্ডজ্ঞা নাই সুযোগ যদি আপুনি যীচুৰ অনুগামী হব খোজে তেন্তে এনেদৰে ঈশ্বৰৰ সৈতে কথা পাতিব পাৰে সেই ঈশ্বৰ মই স্বীকাৰ কৰিছো যে মই এজন পাপী হওঁ মই বিশ্বাস কৰিছো যে যীচুৱে মোৰ পাপৰ বেজ দিবলৈ খোচত মৃত্যুবৰণ কৰিলে মোক ক্ষমা কৰা আৰু শুদ্ধসূচী কৰা আৰু মোক তোমাৰ পৰিয়ালৰ এজন সদস্য কৰি লোৱা মই যীচুৰ পটৰ অনুগামী হ'ব বিচাৰো আৰু মৃত্যুৰ পিছত মই তোমাৰ সৈতে স্বৰ্গ জীয়াই থাকিব বিচাৰো সেই ঈশ্বৰ ধন্যবাদ


In [ ]:
# 1. Install Required Python Packages
!pip3 install -q fasttext transformers

# 2. Setup Directories and Weights
import os
import sys
import json

base_path = "/content/IndicLID/Inference/ai4bharat"

if not os.path.exists(base_path):
    %cd /content
    !git clone https://github.com/AI4Bharat/IndicLID.git

# Setup the models folder inside the directory where IndicLID.py is
%cd {base_path}
!mkdir -p models
%cd models

weights = {
    'indiclid-bert.zip': 'https://github.com/AI4Bharat/IndicLID/releases/download/v1.0/indiclid-bert.zip',
    'indiclid-ftn.zip': 'https://github.com/AI4Bharat/IndicLID/releases/download/v1.0/indiclid-ftn.zip',
    'indiclid-ftr.zip': 'https://github.com/AI4Bharat/IndicLID/releases/download/v1.0/indiclid-ftr.zip'
}

for name, url in weights.items():
    if not os.path.exists(name):
        print(f"Downloading {name}...")
        !wget -q {url}
        !unzip -q -o {name}

# 3. Import and Run Detection
%cd {base_path}
if base_path not in sys.path:
    sys.path.append(base_path)

# Import the class from IndicLID.py
from IndicLID import IndicLID

# Mapping for IndicLID tags to full names
LID_NAME_MAP = {
    "asm_Beng": "Assamese", "ben_Beng": "Bengali", "brx_Deva": "Bodo", "doi_Deva": "Dogri",
    "eng_Latn": "English", "guj_Gujr": "Gujarati", "hin_Deva": "Hindi", "kan_Knda": "Kannada",
    "kas_Arab": "Kashmiri", "gom_Deva": "Konkani", "mai_Deva": "Maithili", "mal_Mlym": "Malayalam",
    "mni_Beng": "Meitei", "mar_Deva": "Marathi", "npi_Deva": "Nepali", "ory_Orya": "Odia",
    "pan_Guru": "Punjabi", "san_Deva": "Sanskrit", "sat_Olck": "Santali", "snd_Arab": "Sindhi",
    "tam_Taml": "Tamil", "tel_Telu": "Telugu", "urd_Arab": "Urdu"
}

try:
    # Initialize
    indic_lid_model = IndicLID(input_threshold=0.5)

    # 'result' comes from the previous transcription cell
    text_to_detect = result
    predictions = indic_lid_model.batch_predict([text_to_detect], batch_size=1)

    print("\n--- IndicLID Detection Result ---")
    if predictions:
        pred_text, lang_script, conf, model_variant = predictions[0]

        # Get readable name
        full_name = LID_NAME_MAP.get(lang_script, "Unknown")

        res = {
            "detected_language": full_name,
            "language_code": lang_script,
            "confidence": float(conf),
            "model_used": model_variant
        }
        print(json.dumps(res, indent=2, ensure_ascii=False))
    else:
        print("No language detected.")
except NameError:
    print("Error: Variable 'result' not found. Please run the transcription cell first.")
except TypeError as e:
    print(f"Error initializing model: {e}")
except ValueError as e:
    print(f"Error unpacking results: {e}.")

# Back to content
%cd /content

/content/IndicLID/Inference/ai4bharat
/content/IndicLID/Inference/ai4bharat/models
/content/IndicLID/Inference/ai4bharat

--- IndicLID Detection Result ---
{
  "detected_language": "Assamese",
  "language_code": "asm_Beng",
  "confidence": 1.0000499486923218,
  "model_used": "IndicLID-FTN"
}
/content


In [ ]:
!pip install -q groq

import os
from groq import Groq
from google.colab import userdata

# Retrieve API key from Colab secrets
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
client = Groq(api_key=GROQ_API_KEY)

# We'll use the file already converted in the previous steps
filename = "converted.wav"

with open(filename, "rb") as file:
    # Using whisper-large-v3-turbo for faster transcription
    transcription = client.audio.transcriptions.create(
      file=(filename, file.read()),
      model="whisper-large-v3",
      response_format="verbose_json",
    )

print(f"Detected Language: {transcription.language}")
print("\n===== Groq Transcript =====")
print(transcription.text)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 9.2 MB/s eta 0:00:00
Detected Language: Bengali

===== Groq Transcript =====
 পাইস্নাং সবি দুতা রাস্তা আসে জিস্যে হিকাইসেল জে দুতা রাস্তা আসে প্রটেক্জন মানুহেই পাপর বহল রাস্থারে হে সোলিসে জিয়ে বিনাখলোই পরিশল না করে। তেম বহল রাস্তা এরি থেক পতেরে সলে জিপতে সর্গলে পরিশল না করে। থেক পতেরে সলা হকলোর কারনে আরু পাপোর দন্দগিযা নাই। মুক ক্যমা করা আর হুদ্ধ হুসি করা আর মুক তুমার পরিযালোর এজন হদেশ করিল্লুআ মাই জিসুর পটোর অনুগামি হবো বিসারো আর মিঠ্যুর �


In [ ]:
!pip install -q sacremoses sentencepiece transformers optimum huggingface-hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 10.8 MB/s eta 0:00:00


In [ ]:
!pip install -q \
transformers \
sentencepiece \
sacremoses \
protobuf \
accelerate

In [ ]:
# Install translation dependencies
!pip install -q sacremoses sentencepiece transformers==4.44.2 optimum

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_IDS = {
    "indic_indic": "ai4bharat/indictrans2-indic-indic-1B",
    "indic_en": "ai4bharat/indictrans2-indic-en-dist-200M",
    "en_indic": "ai4bharat/indictrans2-en-indic-dist-200M",
}

print("Loading IndicTrans2 models...")

tokenizers = {}
translation_models = {}

for name, model_id in MODEL_IDS.items():
    print(f"Loading {model_id}")
    tokenizers[name] = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=True
    )
    translation_models[name] = AutoModelForSeq2SeqLM.from_pretrained(
        model_id,
        trust_remote_code=True
    ).to(device)

In [ ]:
INDIC_LANGS = {
    "as","bn","brx","doi","gu","hi","kn","ks","kok",
    "mai","ml","mni","mr","ne","or","pa","sa",
    "sat","sd","ta","te","ur"
}

TAG_MAP = {
    "as":"asm_Beng",
    "bn":"ben_Beng",
    "brx":"brx_Deva",
    "doi":"doi_Deva",
    "en":"eng_Latn",
    "gu":"guj_Gujr",
    "hi":"hin_Deva",
    "kn":"kan_Knda",
    "ks":"kas_Arab",
    "kok":"gom_Deva",
    "mai":"mai_Deva",
    "ml":"mal_Mlym",
    "mni":"mni_Beng",
    "mr":"mar_Deva",
    "ne":"npi_Deva",
    "or":"ory_Orya",
    "pa":"pan_Guru",
    "sa":"san_Deva",
    "sat":"sat_Olck",
    "sd":"snd_Arab",
    "ta":"tam_Taml",
    "te":"tel_Telu",
    "ur":"urd_Arab"
}

In [ ]:
def translate(text, source_lang, target_lang):
    if source_lang == target_lang:
        return text

    if source_lang == "en" and target_lang in INDIC_LANGS:
        model_key = "en_indic"
    elif source_lang in INDIC_LANGS and target_lang == "en":
        model_key = "indic_en"
    elif source_lang in INDIC_LANGS and target_lang in INDIC_LANGS:
        model_key = "indic_indic"
    else:
        raise ValueError(f"Unsupported translation: {source_lang} -> {target_lang}")

    tokenizer = tokenizers[model_key]
    model = translation_models[model_key]

    src_tag = TAG_MAP[source_lang]
    tgt_tag = TAG_MAP[target_lang]

    inputs = tokenizer(
        text,
        src_lang=src_tag,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        generated = model.generate(
            **inputs,
            tgt_lang=tgt_tag,
            max_new_tokens=512
        )

    return tokenizer.batch_decode(
        generated,
        skip_special_tokens=True
    )[0]

print(f"Detected language: {language}")
print(result)

target_language=input("Enter the language to which i need to get translated")

translated_text = translate(result, language, target_language)

print("\n==============================")
print(f"Translation ({language} -> {target_language})")
print("==============================")
print(translated_text)

Detected language: as


KeyboardInterrupt: Interrupted by user

In [ ]:
# Install translation dependencies
!pip install -q sacremoses sentencepiece transformers==4.44.2 optimum

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_IDS = {
    "indic_indic": "ai4bharat/indictrans2-indic-indic-1B",
    "indic_en": "ai4bharat/indictrans2-indic-en-dist-200M",
    "en_indic": "ai4bharat/indictrans2-en-indic-dist-200M",
}

print("Loading IndicTrans2 models...")

tokenizers = {}
translation_models = {}

for name, model_id in MODEL_IDS.items():
    print(f"Loading {model_id}")
    tokenizers[name] = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=True
    )
    translation_models[name] = AutoModelForSeq2SeqLM.from_pretrained(
        model_id,
        trust_remote_code=True
    ).to(device)

INDIC_LANGS = {
    "as","bn","brx","doi","gu","hi","kn","ks","kok",
    "mai","ml","mni","mr","ne","or","pa","sa",
    "sat","sd","ta","te","ur"
}

TAG_MAP = {
    "as":"asm_Beng",
    "bn":"ben_Beng",
    "brx":"brx_Deva",
    "doi":"doi_Deva",
    "en":"eng_Latn",
    "gu":"guj_Gujr",
    "hi":"hin_Deva",
    "kn":"kan_Knda",
    "ks":"kas_Arab",
    "kok":"gom_Deva",
    "mai":"mai_Deva",
    "ml":"mal_Mlym",
    "mni":"mni_Beng",
    "mr":"mar_Deva",
    "ne":"npi_Deva",
    "or":"ory_Orya",
    "pa":"pan_Guru",
    "sa":"san_Deva",
    "sat":"sat_Olck",
    "sd":"snd_Arab",
    "ta":"tam_Taml",
    "te":"tel_Telu",
    "ur":"urd_Arab"
}

def translate(text, source_lang, target_lang):
    if source_lang == target_lang:
        return text

    if source_lang == "en" and target_lang in INDIC_LANGS:
        model_key = "en_indic"
    elif source_lang in INDIC_LANGS and target_lang == "en":
        model_key = "indic_en"
    elif source_lang in INDIC_LANGS and target_lang in INDIC_LANGS:
        model_key = "indic_indic"
    else:
        raise ValueError(f"Unsupported translation: {source_lang} -> {target_lang}")

    tokenizer = tokenizers[model_key]
    model = translation_models[model_key]

    src_tag = TAG_MAP[source_lang]
    tgt_tag = TAG_MAP[target_lang]

    inputs = tokenizer(
        text,
        src_lang=src_tag,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        generated = model.generate(
            **inputs,
            tgt_lang=tgt_tag,
            max_new_tokens=512
        )

    return tokenizer.batch_decode(
        generated,
        skip_special_tokens=True
    )[0]

print(f"Detected language: {language}")
print(result)

target_language=input("Enter the language to which i need to get translated")

translated_text = translate(result, language, target_language)

print("\n==============================")
print(f"Translation ({language} -> {target_language})")
print("==============================")
print(translated_text)

In [ ]:
# ============================================================
# Text-to-Speech using AI4Bharat Indic Parler-TTS
# ============================================================

!pip -q install git+https://github.com/huggingface/parler-tts.git
!pip -q install soundfile

import torch
import soundfile as sf
from transformers import AutoTokenizer
from parler_tts import ParlerTTSForConditionalGeneration
from IPython.display import Audio

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading Indic Parler-TTS...")

tts_model = ParlerTTSForConditionalGeneration.from_pretrained(
    "ai4bharat/indic-parler-tts"
).to(device)

prompt_tokenizer = AutoTokenizer.from_pretrained(
    "ai4bharat/indic-parler-tts"
)

description_tokenizer = AutoTokenizer.from_pretrained(
    tts_model.config.text_encoder._name_or_path
)

BEST_SPEAKERS = {
    "as": "Amit",
    "bn": "Arjun",
    "brx": "Bikram",
    "doi": "Karan",
    "en": "Thoma",
    "gu": "Yash",
    "hi": "Rohit",
    "kn": "Suresh",
    "kok": "Yash",
    "mai": "Rohit",
    "ml": "Anjali",
    "mni": "Laishram",
    "mr": "Sanjay",
    "ne": "Amrita",
    "or": "Manas",
    "pa": "Divjot",
    "sa": "Aryan",
    "sat": "Aryan",
    "sd": "Rohit",
    "ta": "Jaya",
    "te": "Prakash",
    "ur": "Rohit"
}

# ------------------------------------------------------------
# Try to locate the final text from previous cells
# ------------------------------------------------------------
if "translated_text" in globals():
    text_to_speak = translated_text
elif "transcription_output" in globals():
    text_to_speak = transcription_output
elif "transcription" in globals():
    text_to_speak = transcription
elif "decoded_text" in globals():
    text_to_speak = decoded_text
elif "text" in globals():
    text_to_speak = text
else:
    raise ValueError("Could not find the final ASR/translation text. Assign it to 'text_to_speak'.")

# ------------------------------------------------------------
# Try to locate detected language
# ------------------------------------------------------------
if "language" in globals():
    lang = language
elif "detected_language" in globals():
    lang = detected_language
elif "lang_code" in globals():
    lang = lang_code
else:
    lang = "en"

speaker = BEST_SPEAKERS.get(lang, "Thoma")

description = (
    f"{speaker}'s voice is clear, expressive, natural, "
    "with moderate speaking rate and pitch. "
    "The recording is of very high quality with almost no background noise."
)

print(f"Language : {lang}")
print(f"Speaker  : {speaker}")
print(f"Text     : {text_to_speak}")

description_inputs = description_tokenizer(
    description,
    return_tensors="pt"
).to(device)

prompt_inputs = prompt_tokenizer(
    text_to_speak,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    audio = tts_model.generate(
        input_ids=description_inputs.input_ids,
        attention_mask=description_inputs.attention_mask,
        prompt_input_ids=prompt_inputs.input_ids,
        prompt_attention_mask=prompt_inputs.attention_mask
    )

audio = audio.cpu().numpy().squeeze()

output_file = "tts_output.wav"

sf.write(output_file, audio, tts_model.config.sampling_rate)

print(f"Saved to {output_file}")

Audio(output_file)
